# Flower Detection — Ultralytics benchmark (YOLOv8, YOLOv9, YOLOv10, YOLO11, YOLO26)

Each family trains for 30 epochs and its epoch-15 and epoch-30 states are evaluated on the held-out
test split. One family per process, so a lost session costs one run instead of five.
Run the cells top to bottom; re-running the training cell resumes at the first unfinished family.

In [ ]:
import shutil, subprocess, sys
from pathlib import Path

WORK = Path("/content") if Path("/content").is_dir() else Path("/kaggle/working")
IN_COLAB = WORK == Path("/content")
PROJECT = WORK / "flower_detect"
REPO = "https://github.com/hit1363/flower_detect.git"
DATA_YAML = PROJECT / "dataset/Annotated/split/data.yaml"
# Annotated images exported from the team Drive folder (images/, labels/, metadata.csv).
DRIVE_SOURCE = Path("/content/drive/MyDrive/flower_detection/02_annotated")
DEVICE, BATCH, WORKERS = "0", 8, 2
print(f"platform={'colab' if IN_COLAB else 'kaggle'} project={PROJECT}")

In [ ]:
!pip install -q ultralytics==8.4.218 pyyaml
import torch
assert torch.cuda.is_available(), "Select a GPU runtime (Runtime > Change runtime type)."
print(torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
if not PROJECT.exists():
    subprocess.run(["git", "clone", "-q", REPO, str(PROJECT)], check=True)
print(subprocess.run(["git", "-C", str(PROJECT), "log", "-1", "--oneline"],
                     capture_output=True, text=True).stdout.strip())

### Dataset
Edit `DRIVE_SOURCE` in the config cell to point at the team's annotated folder.

In [ ]:
# Pull the annotated source from Drive once, then validate and split it locally.
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
SOURCE = PROJECT / "dataset/Annotated/source"
if DRIVE_SOURCE.is_dir():
    for folder in ("images", "labels"):
        shutil.copytree(DRIVE_SOURCE / folder, SOURCE / folder, dirs_exist_ok=True)
    shutil.copy2(DRIVE_SOURCE / "metadata.csv", SOURCE / "metadata.csv")
    print(f"Copied annotated source from {DRIVE_SOURCE}")
else:
    print(f"No Drive source at {DRIVE_SOURCE}; using {SOURCE} as already synced.")

In [ ]:
# validate first, then split; split refuses to overwrite an existing split on purpose
subprocess.run([sys.executable, "scripts/prepare_dataset.py", "validate"], cwd=PROJECT, check=True)
subprocess.run([sys.executable, "scripts/prepare_dataset.py", "split"], cwd=PROJECT, check=True)

In [ ]:
subprocess.run([sys.executable, "scripts/train_ultralytics.py", "--data-yaml", str(DATA_YAML),
                "--models", "yolov8", "--preflight"], cwd=PROJECT, check=True)

In [ ]:
for family in ["yolov8", "yolov9", "yolov10", "yolo11", "yolo26"]:
    run_dir = PROJECT / "supplementary" / "runs" / family
    if (run_dir / "results.csv").is_file():
        print(f"{family}: already evaluated, skipping")
        continue
    if run_dir.exists():
        print(f"{family}: unfinished run folder {run_dir}; delete it to retry this family")
        continue
    subprocess.run([sys.executable, "scripts/train_ultralytics.py", "--data-yaml", str(DATA_YAML),
                    "--models", family, "--device", DEVICE, "--batch", str(BATCH),
                    "--workers", str(WORKERS)], cwd=PROJECT, check=True)

In [ ]:
import pandas as pd
results = pd.read_csv(PROJECT / "supplementary/benchmark_results.csv")
print(f"{len(results)} of 12 rows recorded")
results

In [ ]:
import zipfile
archive = WORK / "flower_supplementary.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted((PROJECT / "supplementary").rglob("*")):
        if path.is_file():
            bundle.write(path, path.relative_to(PROJECT))
print(f"Packed {archive} ({archive.stat().st_size / 1e6:.1f} MB)")
if IN_COLAB:
    target = Path("/content/drive/MyDrive/flower_detection")
    target.mkdir(parents=True, exist_ok=True)
    shutil.copy2(archive, target / archive.name)
    shutil.copy2(PROJECT / "supplementary/benchmark_results.csv", target / "benchmark_results.csv")
    print(f"Copied results to {target}")